# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0369/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


Signal 1: CTR vs Position
I want to check whether pages with stronger search positions but relatively low CTR show a useful pattern for identifying possible CTR improvement opportunities.

Verdict: MIXED
The CTR-versus-position pattern is observed in the March 2026 data. A substantial number of rows have a strong search position combined with low CTR. However, the bucket counts alone do not show whether these rows would consistently benefit from an action, so I treat the signal as mixed rather than confirmed.

Signal 2: Volume
I want to check whether pages with higher search volume provide a useful signal for identifying quick-win opportunities.

Verdict: MIXED
The volume signal is clearly observed in the March 2026 data, with most rows having low impressions and a smaller group having medium or high impressions. However, the bucket counts alone do not show that high-volume content is more likely to be a useful quick-win opportunity, so I treat the signal as mixed rather than confirmed.


*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")



Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

ctr_position = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN 'No impressions'
        WHEN gsc_avg_position <= 3 AND (gsc_clicks * 100.0 / gsc_impressions) < 1
            THEN 'Top position + low CTR'
        WHEN gsc_avg_position <= 10 AND (gsc_clicks * 100.0 / gsc_impressions) < 2
            THEN 'Page 1 + low CTR'
        WHEN gsc_avg_position <= 10
            THEN 'Page 1'
        ELSE 'Lower position'
    END AS bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY bucket
ORDER BY n DESC
""").df()

ctr_position


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n
0,Lower position,1427577
1,Page 1 + low CTR,1427239
2,Top position + low CTR,680844
3,Page 1,75401


In [5]:
volume_check = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN 'No impressions'
        WHEN gsc_impressions < 100 THEN 'Low volume'
        WHEN gsc_impressions < 1000 THEN 'Medium volume'
        ELSE 'High volume'
    END AS bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY bucket
ORDER BY
    CASE bucket
        WHEN 'No impressions' THEN 1
        WHEN 'Low volume' THEN 2
        WHEN 'Medium volume' THEN 3
        WHEN 'High volume' THEN 4
    END
""").df()

volume_check

,bucket,n
0,Low volume,2972453
1,Medium volume,606189
2,High volume,32419


In [6]:
summary = con.sql(f"""
SELECT
    COUNT(*) AS n,
    ROUND(AVG(gsc_avg_position), 2) AS avg_position,
    ROUND(AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 100.0 / gsc_impressions
        END
    ), 2) AS avg_ctr,
    ROUND(AVG(gsc_impressions), 2) AS avg_impressions
FROM {REL}
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,avg_position,avg_ctr,avg_impressions
0,3611061,15.83,0.31,77.72


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.